# AML Detection: Feature Engineering & Ordered Target Encoding

**Objective.** Compare CatBoost's native categorical handling with a timestamp-aware **Ordered Target Encoder (OTE)** for a highly imbalanced, time-ordered AML classification problem.

### Experiment design
- chronological transaction data with equal timestamps treated as one information batch;
- transaction-level and strictly historical behavioral features;
- bank-scoped account identities;
- negative-class downsampling with inverse-probability weighting;
- Native CatBoost and OTE trained on the **same sampled rows and weights**;
- **PR-AUC** as the primary metric;
- validation for model comparison and early stopping;
- test set used only for final evaluation.

> **Leakage constraint:** historical and custom target-based features for a row may use only information from timestamps strictly earlier than that row's timestamp.

## 1. Setup

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from catboost import CatBoostClassifier

from aml.data.load import load_raw_data


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_OUTPUT_DIR = DATA_DIR

## 2. Raw Data Validation

Before feature generation, the raw transaction table is checked for:
- required schema;
- chronological ordering;
- binary target integrity.

The summary also makes the class imbalance explicit, which motivates PR-AUC and weighted downsampling later in the experiment.

In [8]:
df = load_raw_data()

required_columns = {
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
}
missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Required columns not found: {sorted(missing_columns)}"
assert df["Timestamp"].is_monotonic_increasing, "Transactions are not sorted by time"
assert set(pd.unique(df["Is Laundering"])).issubset({0, 1}), "The target must be binary"

pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "positives": int(df["Is Laundering"].sum()),
        "prevalence": df["Is Laundering"].mean(),
        "start": df["Timestamp"].min(),
        "end": df["Timestamp"].max(),
    },
    name="raw data",
)

rows                      6924041
columns                        15
positives                    3565
prevalence               0.000515
start         2022-09-01 00:00:00
end           2022-09-17 15:28:00
Name: raw data, dtype: object

## 3. Feature Engineering

Feature engineering is split into two groups:

1. **Current-transaction features** — derived only from the transaction being scored.
2. **Historical features** — derived from events strictly preceding the current transaction.

### 3.1 Current-Transaction Features

These features use only information available at scoring time:
- cyclical hour/day-of-week representations;
- night/weekend indicators;
- bank, account, and currency consistency flags;
- log-transformed payment/receipt amounts;
- amount discrepancy;
- round-value indicators.

In [9]:
def basic_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.drop(columns=["Timestamp", "Is Laundering"]).reset_index(drop=True).copy()
    timestamp = df["Timestamp"].reset_index(drop=True)

    # Time
    hour = timestamp.dt.hour
    day_of_week = timestamp.dt.dayofweek
    paid = df["Amount Paid"].clip(lower=0).reset_index(drop=True)
    received = df["Amount Received"].clip(lower=0).reset_index(drop=True)

    X["hour"] = hour.astype("int8")
    X["dayofweek"] = day_of_week.astype("int8")
    X["day"] = timestamp.dt.day.astype("int8")
    X["month"] = timestamp.dt.month.astype("int8")

    X["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
    X["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")
    X["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7).astype("float32")
    X["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7).astype("float32")
    X["is_night"] = hour.between(0, 5).astype("int8")
    X["is_weekend"] = day_of_week.isin([5, 6]).astype("int8")

    # Current-transaction consistency
    X["is_currency_same"] = (
        df["Receiving Currency"] == df["Payment Currency"]
    ).astype("int8").to_numpy()
    X["is_same_bank"] = (
        df["From Bank"] == df["To Bank"]
    ).astype("int8").to_numpy()
    X["is_self_transfer"] = (
        (df["From Bank"] == df["To Bank"])
        & (df["Account"] == df["Account.1"])
    ).astype("int8").to_numpy()

    # Amount transformations
    X["log_amount_paid"] = np.log1p(paid).astype("float32")
    X["log_amount_received"] = np.log1p(received).astype("float32")
    X["amount_log_gap"] = np.abs(
        X["log_amount_paid"] - X["log_amount_received"]
    ).astype("float32")
    X["is_round_10"] = np.isclose(np.mod(paid, 10), 0, atol=1e-8).astype("int8")
    X["is_round_100"] = np.isclose(np.mod(paid, 100), 0, atol=1e-8).astype("int8")
    X["is_round_1000"] = np.isclose(np.mod(paid, 1_000), 0, atol=1e-8).astype("int8")

    return X


timestamp = df["Timestamp"].reset_index(drop=True)
X = basic_features(df)
X.shape

(6924041, 28)

### 3.2 Historical Behavioral Features

Historical features are calculated on the full chronological sequence **before the temporal split**.

Two constraints are enforced:
- account identity is bank-scoped: `(From Bank, Account)` for senders and `(To Bank, Account.1)` for receivers;
- all rows sharing the current timestamp are excluded from the history, not merely the current row.

This prevents same-timestamp leakage in cumulative counts, sums, time-gap features, counterparty history, and amount statistics. The target is never used in these features.

In [10]:
def _composite_key(X: pd.DataFrame, columns) -> pd.Series:
    """Stable compact key for bank-scoped account identities and interactions."""
    return pd.util.hash_pandas_object(
        X.loc[:, list(columns)],
        index=False,
    ).astype("uint64")


def _strict_prior_count(entity: pd.Series, time_group: pd.Series) -> pd.Series:
    """Number of entity rows from strictly earlier timestamp groups."""
    row_count_before = entity.groupby(entity, sort=False).cumcount()
    same_time_before = entity.groupby([entity, time_group], sort=False).cumcount()
    return (row_count_before - same_time_before).astype("int32")


def _strict_prior_cumsum(
    values: pd.Series,
    entity: pd.Series,
    time_group: pd.Series,
) -> pd.Series:
    """Cumulative sum over strictly earlier timestamp groups."""
    entity_inclusive = values.groupby(entity, sort=False).cumsum()
    same_time_inclusive = values.groupby([entity, time_group], sort=False).cumsum()
    return entity_inclusive - same_time_inclusive


def _strict_previous_timestamp(
    timestamp: pd.Series,
    entity: pd.Series,
) -> pd.Series:
    """Previous distinct timestamp; all rows in one timestamp batch share the same history."""
    previous_row_timestamp = timestamp.groupby(entity, sort=False).shift()
    previous_strict = previous_row_timestamp.where(previous_row_timestamp < timestamp)
    return previous_strict.groupby(entity, sort=False).ffill()


def historical_features(
    X: pd.DataFrame,
    timestamp: pd.Series,
) -> pd.DataFrame:
    X = X.reset_index(drop=True).copy()
    timestamp = pd.Series(timestamp).reset_index(drop=True)

    if len(X) != len(timestamp):
        raise ValueError("X and timestamp must have the same number of rows")
    if not timestamp.is_monotonic_increasing:
        raise ValueError("timestamp must be sorted in ascending order")

    # Equal timestamps form one information batch: no row may see another row in the same batch.
    time_group = timestamp.ne(timestamp.shift()).cumsum().astype("int32")

    # Account numbers are treated as bank-scoped identities.
    sender = _composite_key(X, ["From Bank", "Account"]).reset_index(drop=True)
    receiver = _composite_key(X, ["To Bank", "Account.1"]).reset_index(drop=True)
    pair = pd.util.hash_pandas_object(
        pd.DataFrame({"sender": sender, "receiver": receiver}),
        index=False,
    ).astype("uint64")

    paid = X["Amount Paid"].clip(lower=0).reset_index(drop=True)

    # Transaction counts from strictly earlier timestamps.
    sender_previous_count = _strict_prior_count(sender, time_group)
    receiver_previous_count = _strict_prior_count(receiver, time_group)
    pair_previous_count = _strict_prior_count(pair, time_group)

    X["sender_prev_tx_log"] = np.log1p(sender_previous_count).astype("float32")
    X["receiver_prev_tx_log"] = np.log1p(receiver_previous_count).astype("float32")
    X["pair_prev_tx_log"] = np.log1p(pair_previous_count).astype("float32")
    X["is_new_pair"] = (pair_previous_count == 0).astype("int8")

    # Unique counterparties established strictly before the current timestamp.
    first_pair_row = pair.groupby(pair, sort=False).cumcount().eq(0).astype("int32")
    sender_unique_receivers_before = _strict_prior_cumsum(
        first_pair_row, sender, time_group
    )
    receiver_unique_senders_before = _strict_prior_cumsum(
        first_pair_row, receiver, time_group
    )
    X["sender_unique_receivers_log"] = np.log1p(
        sender_unique_receivers_before
    ).astype("float32")
    X["receiver_unique_senders_log"] = np.log1p(
        receiver_unique_senders_before
    ).astype("float32")

    # Historical interbank share.
    is_interbank = (
        X["From Bank"] != X["To Bank"]
    ).astype("int32").reset_index(drop=True)
    sender_prior_interbank_count = _strict_prior_cumsum(
        is_interbank, sender, time_group
    )
    X["sender_prior_interbank_ratio"] = (
        sender_prior_interbank_count
        / sender_previous_count.replace(0, np.nan)
    ).fillna(0).astype("float32")

    # Time since the previous distinct activity timestamp.
    sender_previous_timestamp = _strict_previous_timestamp(timestamp, sender)
    receiver_previous_timestamp = _strict_previous_timestamp(timestamp, receiver)

    sender_minutes_since_previous = (
        (timestamp - sender_previous_timestamp).dt.total_seconds() / 60
    )
    receiver_minutes_since_previous = (
        (timestamp - receiver_previous_timestamp).dt.total_seconds() / 60
    )
    X["sender_minutes_since_prev_log"] = np.log1p(
        sender_minutes_since_previous.clip(lower=0)
    ).fillna(-1).astype("float32")
    X["receiver_minutes_since_prev_log"] = np.log1p(
        receiver_minutes_since_previous.clip(lower=0)
    ).fillna(-1).astype("float32")

    # Average interval between distinct historical sender activity timestamps.
    # The current timestamp's interval is excluded, and the denominator counts
    # actual historical intervals rather than historical transactions.
    sender_previous_row_timestamp = timestamp.groupby(sender, sort=False).shift()
    sender_batch_start = sender_previous_row_timestamp.ne(timestamp)
    sender_interval_minutes = sender_minutes_since_previous.where(sender_batch_start)

    sender_prior_interval_sum = _strict_prior_cumsum(
        sender_interval_minutes.fillna(0.0),
        sender,
        time_group,
    )
    sender_prior_interval_count = _strict_prior_cumsum(
        sender_interval_minutes.notna().astype("int32"),
        sender,
        time_group,
    )
    sender_prior_average_minutes = (
        sender_prior_interval_sum
        / sender_prior_interval_count.replace(0, np.nan)
    )
    X["sender_prior_avg_minutes_log"] = np.log1p(
        sender_prior_average_minutes.clip(lower=0)
    ).fillna(-1).astype("float32")

    # Amount anomaly relative to the sender's strictly historical mean amount.
    sender_prior_amount_sum = _strict_prior_cumsum(paid, sender, time_group)
    sender_prior_amount_mean = (
        sender_prior_amount_sum
        / sender_previous_count.replace(0, np.nan)
    )
    amount_to_prior_mean = paid / sender_prior_amount_mean.replace(0, np.nan)
    X["amount_to_sender_prior_mean_log"] = (
        np.log1p(amount_to_prior_mean.clip(lower=0, upper=1e6))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .astype("float32")
    )

    return X


X = historical_features(X, timestamp)
X.shape

(6924041, 39)

## 4. Temporal Splits & Class-Imbalance Strategy

The engineered matrix `X` is split directly into chronological **train / validation / test** periods.

Target proportions remain approximately **70% / 15% / 15%**, but split boundaries are snapped to the beginning of a timestamp batch. Therefore a timestamp can never be divided across two splits, and the periods satisfy:

`train time < validation time < test time`

To reduce training cost:
- all positive training rows are retained;
- 30% of negative training rows are sampled;
- sampled negatives receive inverse-probability weights;
- sampled rows are restored to chronological order before fitting.

Crucially, **Native CatBoost and OTE use exactly the same sampled rows, labels, and weights**. OTE does not access labels from discarded training rows.

In [11]:
y = df["Is Laundering"].astype("int8").reset_index(drop=True)

assert len(X) == len(y) == len(timestamp), "X, y, and timestamp must align"

train_fraction = 0.70
val_fraction = 0.15
n_rows = len(X)

def timestamp_safe_boundary(timestamp: pd.Series, fraction: float) -> int:
    """Snap a target row fraction to the start of its timestamp batch."""
    n = len(timestamp)
    target_position = min(max(int(n * fraction), 1), n - 1)
    boundary_timestamp = timestamp.iloc[target_position]
    return int(timestamp.searchsorted(boundary_timestamp, side="left"))


train_end = timestamp_safe_boundary(timestamp, train_fraction)
val_end = timestamp_safe_boundary(timestamp, train_fraction + val_fraction)

assert 0 < train_end < val_end < n_rows, "Temporal split produced an empty partition"

X_train = X.iloc[:train_end].reset_index(drop=True)
y_train = y.iloc[:train_end].reset_index(drop=True)
timestamp_train = timestamp.iloc[:train_end].reset_index(drop=True)

X_val = X.iloc[train_end:val_end].reset_index(drop=True)
y_val = y.iloc[train_end:val_end].reset_index(drop=True)
timestamp_val = timestamp.iloc[train_end:val_end].reset_index(drop=True)

X_test = X.iloc[val_end:].reset_index(drop=True)
y_test = y.iloc[val_end:].reset_index(drop=True)
timestamp_test = timestamp.iloc[val_end:].reset_index(drop=True)

assert timestamp_train.iloc[-1] < timestamp_val.iloc[0]
assert timestamp_val.iloc[-1] < timestamp_test.iloc[0]

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_val), len(X_test)],
        "positives": [y_train.sum(), y_val.sum(), y_test.sum()],
        "prevalence": [y_train.mean(), y_val.mean(), y_test.mean()],
        "start": [
            timestamp_train.iloc[0],
            timestamp_val.iloc[0],
            timestamp_test.iloc[0],
        ],
        "end": [
            timestamp_train.iloc[-1],
            timestamp_val.iloc[-1],
            timestamp_test.iloc[-1],
        ],
    },
    index=["train", "validation", "test"],
)

,rows,positives,prevalence,start,end
train,4846503,2231,0.000460,2022-09-01 00:00:00,2022-09-07 14:47:00
validation,1038436,583,0.000561,2022-09-07 14:48:00,2022-09-09 03:12:00
test,1039102,751,0.000723,2022-09-09 03:13:00,2022-09-17 15:28:00


In [6]:
y_array = y_train.to_numpy()
positive_positions = np.flatnonzero(y_array == 1)
negative_positions = np.flatnonzero(y_array == 0)

rng = np.random.default_rng(42)
negative_sample_size = max(1, int(0.3 * len(negative_positions)))
sampled_negative_positions = rng.choice(
    negative_positions,
    size=negative_sample_size,
    replace=False,
)

fit_positions = np.concatenate([positive_positions, sampled_negative_positions])
rng.shuffle(fit_positions)

X_fit = X_train.iloc[fit_positions].reset_index(drop=True)
y_fit = y_train.iloc[fit_positions].reset_index(drop=True)
negative_weight = len(negative_positions) / negative_sample_size
sample_weight = np.where(y_fit.to_numpy() == 0, negative_weight, 1.0)

In [12]:
y_array = y_train.to_numpy()
positive_positions = np.flatnonzero(y_array == 1)
negative_positions = np.flatnonzero(y_array == 0)

rng = np.random.default_rng(42)
negative_sample_size = max(1, int(0.3 * len(negative_positions)))
sampled_negative_positions = rng.choice(
    negative_positions,
    size=negative_sample_size,
    replace=False,
)

# Both models see the same rows in the same chronological order.
fit_positions = np.sort(
    np.concatenate([positive_positions, sampled_negative_positions])
)

X_fit = X_train.iloc[fit_positions].reset_index(drop=True)
y_fit = y_train.iloc[fit_positions].reset_index(drop=True)
timestamp_fit = timestamp_train.iloc[fit_positions].reset_index(drop=True)

negative_weight = len(negative_positions) / negative_sample_size
sample_weight = np.where(
    y_fit.to_numpy() == 0,
    negative_weight,
    1.0,
).astype("float64")

assert timestamp_fit.is_monotonic_increasing
assert len(X_fit) == len(y_fit) == len(timestamp_fit) == len(sample_weight)

pd.Series(
    {
        "fit_rows": len(X_fit),
        "fit_positives": int(y_fit.sum()),
        "negative_sampling_rate": negative_sample_size / len(negative_positions),
        "negative_weight": negative_weight,
    },
    name="training subset",
)

fit_rows                  1.455512e+06
fit_positives             2.231000e+03
negative_sampling_rate    2.999999e-01
negative_weight           3.333335e+00
Name: training subset, dtype: float64

## 5. Baseline: Native CatBoost Categoricals

Native CatBoost is the reference model.

For comparability:
- categorical columns are declared explicitly rather than inferred from pandas dtypes;
- the same sampled rows and inverse-probability weights are used as for OTE;
- the sampled training rows remain chronological;
- `has_time=True` disables CatBoost's random object permutations and preserves input order;
- model hyperparameters, validation set, and early stopping are shared with the OTE model.

The custom OTE additionally enforces timestamp-batch semantics explicitly, including equal-timestamp ties.

In [13]:
categorical_features = [
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Receiving Currency",
    "Payment Currency",
    "Payment Format",
]

missing_categorical = set(categorical_features) - set(X_fit.columns)
assert not missing_categorical, f"Categorical columns missing: {sorted(missing_categorical)}"

model_params = dict(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=6.0,
    random_seed=42,
    has_time=True,
    allow_writing_files=False,
)

cat_native = CatBoostClassifier(**model_params)
cat_native.fit(
    X_fit,
    y_fit,
    sample_weight=sample_weight,
    eval_set=(X_val, y_val),
    early_stopping_rounds=150,
    cat_features=categorical_features,
    verbose=100,
)

native_val_score = cat_native.predict_proba(X_val)[:, 1]
catboost_native_pr_auc = average_precision_score(y_val, native_val_score)
catboost_native_pr_auc

0:	learn: 0.0018553	test: 0.0021888	best: 0.0021888 (0)	total: 1.12s	remaining: 55m 47s
100:	learn: 0.1939922	test: 0.1651552	best: 0.1657558 (99)	total: 1m 14s	remaining: 35m 35s
200:	learn: 0.2426384	test: 0.2009661	best: 0.2009661 (200)	total: 2m 36s	remaining: 36m 14s
300:	learn: 0.2564232	test: 0.2062126	best: 0.2062302 (299)	total: 3m 54s	remaining: 35m 1s
400:	learn: 0.2639763	test: 0.2080177	best: 0.2080177 (400)	total: 5m 4s	remaining: 32m 51s
500:	learn: 0.2723063	test: 0.2124714	best: 0.2124714 (500)	total: 6m 24s	remaining: 31m 57s
600:	learn: 0.2826926	test: 0.2196301	best: 0.2197797 (581)	total: 7m 43s	remaining: 30m 49s
700:	learn: 0.2889413	test: 0.2213012	best: 0.2216264 (667)	total: 9m 3s	remaining: 29m 41s
800:	learn: 0.2940735	test: 0.2221096	best: 0.2221132 (797)	total: 10m 16s	remaining: 28m 13s
900:	learn: 0.3002451	test: 0.2241610	best: 0.2241967 (898)	total: 11m 47s	remaining: 27m 28s
1000:	learn: 0.3063826	test: 0.2255582	best: 0.2255582 (1000)	total: 13m 16s	

0.2277813691490637

## 6. Ordered Target Encoder

The custom encoder is timestamp-aware.

For a training row at time \(t\), target statistics use only sampled training labels with timestamps **strictly smaller than \(t\)**. Rows with the same timestamp are encoded from the same prior history and never reveal labels to one another.

The encoder also consumes the same inverse-probability weights used by CatBoost. This corrects the target-rate statistics for negative downsampling without giving OTE access to discarded training labels.

For each categorical key it creates:
- ordered weighted target rates at multiple smoothing strengths;
- `log1p` weighted historical support;
- selected low-cost categorical interactions.

Validation and test rows use maps fitted only on the sampled training period.

### 6.1 Encoder Implementation

The implementation explicitly subtracts the complete current timestamp batch from cumulative target statistics. This is stronger than merely excluding the current row.

In [14]:

class OrderedTargetEncoder:
    """
    Timestamp-aware ordered target encoder for binary classification.

    Training rows:
        TE(x_i) uses only labels from rows with timestamp < timestamp_i.
        Rows sharing the same timestamp never use one another's targets.

    Validation/test rows:
        use statistics fitted on the complete sampled training period.

    Downsampling correction:
        optional sample weights are used inside the target statistics, so the
        encoded rates remain aligned with the original class distribution.
    """

    def __init__(
        self,
        categorical_features,
        interactions=None,
        smoothing=(20.0, 200.0),
        add_log_count=True,
        initial_prior=0.0,
    ):
        self.categorical_features = list(categorical_features)
        self.interactions = [tuple(cols) for cols in (interactions or [])]
        self.smoothing = tuple(float(alpha) for alpha in smoothing)
        self.add_log_count = bool(add_log_count)
        self.initial_prior = float(initial_prior)

    @staticmethod
    def _feature_name(columns):
        return "__".join(
            str(col).replace(" ", "_").replace(".", "_")
            for col in columns
        )

    @staticmethod
    def _key(X, columns):
        return pd.util.hash_pandas_object(
            X.loc[:, list(columns)],
            index=False,
        ).astype("uint64")

    def _specs(self):
        specs = [(col,) for col in self.categorical_features] + self.interactions
        return list(dict.fromkeys(specs))

    @staticmethod
    def _validate_timestamp(timestamp, n_rows):
        timestamp = pd.Series(timestamp).reset_index(drop=True)
        if len(timestamp) != n_rows:
            raise ValueError("X and timestamp must have the same number of rows")
        if not timestamp.is_monotonic_increasing:
            raise ValueError("timestamp must be sorted in ascending order")
        return timestamp

    @staticmethod
    def _weights(sample_weight, n_rows):
        if sample_weight is None:
            return np.ones(n_rows, dtype=np.float64)

        weight = np.asarray(sample_weight, dtype=np.float64)
        if len(weight) != n_rows:
            raise ValueError("sample_weight must have the same number of rows as X")
        if np.any(weight <= 0):
            raise ValueError("sample_weight must be strictly positive")
        return weight

    def _fit_maps(self, X, y_array, weight_array):
        self.specs_ = self._specs()
        weighted_target = y_array * weight_array
        self.global_prior_ = float(
            weighted_target.sum() / weight_array.sum()
        )
        self.stats_ = {}

        for spec in self.specs_:
            key = self._key(X, spec).to_numpy()
            stats_frame = pd.DataFrame(
                {
                    "key": key,
                    "weighted_target": weighted_target,
                    "weight": weight_array,
                }
            )
            self.stats_[spec] = (
                stats_frame
                .groupby("key", sort=False, observed=True)
                .agg(
                    sum=("weighted_target", "sum"),
                    count=("weight", "sum"),
                )
            )

    def fit(self, X, y, sample_weight=None):
        y_array = np.asarray(y, dtype=np.float64)
        if len(X) != len(y_array):
            raise ValueError("X and y must have the same number of rows")

        weight_array = self._weights(sample_weight, len(X))
        self._fit_maps(X, y_array, weight_array)
        return self

    def fit_transform(self, X, y, timestamp, sample_weight=None):
        y_array = np.asarray(y, dtype=np.float64)
        n_rows = len(X)

        if n_rows != len(y_array):
            raise ValueError("X and y must have the same number of rows")

        timestamp = self._validate_timestamp(timestamp, n_rows)
        weight_array = self._weights(sample_weight, n_rows)
        time_group = timestamp.ne(timestamp.shift()).cumsum().astype("int32")

        target = pd.Series(y_array, copy=False)
        weight = pd.Series(weight_array, copy=False)
        weighted_target = target * weight

        # Weighted global prior from strictly earlier timestamp groups.
        global_target_before = (
            weighted_target.cumsum()
            - weighted_target.groupby(time_group, sort=False).cumsum()
        ).to_numpy(dtype=np.float64)
        global_weight_before = (
            weight.cumsum()
            - weight.groupby(time_group, sort=False).cumsum()
        ).to_numpy(dtype=np.float64)

        global_prior_before = np.divide(
            global_target_before,
            global_weight_before,
            out=np.full(n_rows, self.initial_prior, dtype=np.float64),
            where=global_weight_before > 0,
        )

        encoded = (
            X.drop(columns=self.categorical_features, errors="ignore")
            .reset_index(drop=True)
            .copy()
        )
        self.specs_ = self._specs()

        for spec in self.specs_:
            key = self._key(X, spec).reset_index(drop=True)

            # Same-timestamp rows are removed as one batch, so no tie leakage is possible.
            category_target_before = (
                weighted_target.groupby(key, sort=False).cumsum()
                - weighted_target.groupby([key, time_group], sort=False).cumsum()
            ).to_numpy(dtype=np.float64)

            category_weight_before = (
                weight.groupby(key, sort=False).cumsum()
                - weight.groupby([key, time_group], sort=False).cumsum()
            ).to_numpy(dtype=np.float64)

            name = self._feature_name(spec)
            for alpha in self.smoothing:
                rate = (
                    category_target_before + alpha * global_prior_before
                ) / (
                    category_weight_before + alpha
                )
                encoded[f"{name}__ote_{alpha:g}"] = rate.astype("float32")

            if self.add_log_count:
                encoded[f"{name}__hist_log_count"] = np.log1p(
                    category_weight_before
                ).astype("float32")

        self._fit_maps(X, y_array, weight_array)
        return encoded

    def transform(self, X):
        if not hasattr(self, "stats_"):
            raise RuntimeError("Call fit(...) or fit_transform(...) before transform(...)")

        encoded = (
            X.drop(columns=self.categorical_features, errors="ignore")
            .reset_index(drop=True)
            .copy()
        )

        for spec in self.specs_:
            key = self._key(X, spec).reset_index(drop=True)
            stats = self.stats_[spec]

            category_sum = (
                key.map(stats["sum"])
                .fillna(0.0)
                .to_numpy(dtype=np.float64)
            )
            category_count = (
                key.map(stats["count"])
                .fillna(0.0)
                .to_numpy(dtype=np.float64)
            )

            name = self._feature_name(spec)
            for alpha in self.smoothing:
                rate = (
                    category_sum + alpha * self.global_prior_
                ) / (
                    category_count + alpha
                )
                encoded[f"{name}__ote_{alpha:g}"] = rate.astype("float32")

            if self.add_log_count:
                encoded[f"{name}__hist_log_count"] = np.log1p(
                    category_count
                ).astype("float32")

        return encoded

### 6.2 Encoding Configuration

Single categorical columns are supplemented with selected interactions. Bank-account interactions preserve account identity without materializing the extremely sparse sender-account × receiver-account cross.

In [15]:
target_encoding_interactions = [
    ("From Bank", "Account"),
    ("To Bank", "Account.1"),
    ("From Bank", "To Bank"),
    ("Receiving Currency", "Payment Currency"),
]

ordered_target_encoder = OrderedTargetEncoder(
    categorical_features=categorical_features,
    interactions=target_encoding_interactions,
    smoothing=(20.0, 200.0),
    add_log_count=True,
)

### 6.3 Encode Training Subset and Validation

OTE is fitted on exactly `X_fit / y_fit / sample_weight`, preserving chronological order and timestamp-batch isolation.

In [16]:
X_fit_ote = ordered_target_encoder.fit_transform(
    X_fit,
    y_fit,
    timestamp_fit,
    sample_weight=sample_weight,
)
X_val_ote = ordered_target_encoder.transform(X_val)

non_numeric_ote = X_fit_ote.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
assert not non_numeric_ote, f"OTE output must be numeric, found: {non_numeric_ote}"
assert list(X_fit_ote.columns) == list(X_val_ote.columns)

X_fit_ote.shape, X_val_ote.shape

((1455512, 65), (1038436, 65))

### 6.4 Train CatBoost on OTE Features

The downstream CatBoost model uses the **same hyperparameters, chronological sampled rows, labels, and sample weights** as the native-categorical baseline. The experimental difference is the categorical representation.

In [17]:
cat_ote = CatBoostClassifier(**model_params)
cat_ote.fit(
    X_fit_ote,
    y_fit,
    sample_weight=sample_weight,
    eval_set=(X_val_ote, y_val),
    early_stopping_rounds=150,
    verbose=100,
)

ote_val_score = cat_ote.predict_proba(X_val_ote)[:, 1]
ordered_te_pr_auc = average_precision_score(y_val, ote_val_score)
ordered_te_pr_auc

0:	learn: 0.0012614	test: 0.0023591	best: 0.0023591 (0)	total: 146ms	remaining: 7m 17s
100:	learn: 0.2287555	test: 0.2402651	best: 0.2402651 (100)	total: 25.3s	remaining: 12m 4s
200:	learn: 0.2739653	test: 0.2626317	best: 0.2626317 (200)	total: 50.8s	remaining: 11m 48s
300:	learn: 0.2950075	test: 0.2638693	best: 0.2643890 (271)	total: 1m 21s	remaining: 12m 6s
400:	learn: 0.3111532	test: 0.2609988	best: 0.2643890 (271)	total: 1m 49s	remaining: 11m 47s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.2643890146
bestIteration = 271

Shrink model to first 272 iterations.


0.26460508992228965

## 7. Validation Comparison

Model selection is based on validation PR-AUC. The table below reports both the absolute and relative uplift from replacing native CatBoost categorical handling with OTE.

In [18]:
pd.Series(
    {
        "CatBoost native categorical": catboost_native_pr_auc,
        "OrderedTargetEncoder": ordered_te_pr_auc,
        "absolute_delta": ordered_te_pr_auc - catboost_native_pr_auc,
        "relative_delta_pct": (
            ordered_te_pr_auc / catboost_native_pr_auc - 1
        ) * 100,
    },
    name="validation PR-AUC",
)

CatBoost native categorical     0.227781
OrderedTargetEncoder            0.264605
absolute_delta                  0.036824
relative_delta_pct             16.166257
Name: validation PR-AUC, dtype: float64

## 8. Final Test Evaluation

The test split is evaluated only after the validation comparison is complete. Both candidate models are measured on the same untouched test set.

### 8.1 Ordered Target Encoder Model

In [19]:
# Use test only after validation-based model comparison/tuning is finished.
X_test_ote = ordered_target_encoder.transform(X_test)
ote_test_score = cat_ote.predict_proba(X_test_ote)[:, 1]
ote_test_pr_auc = average_precision_score(y_test, ote_test_score)
ote_test_pr_auc

0.31962101220588157

### 8.2 Native CatBoost Baseline

In [20]:
native_test_score = cat_native.predict_proba(X_test)[:, 1]
native_test_pr_auc = average_precision_score(y_test, native_test_score)
native_test_pr_auc

0.3071830466877899

## 9. Results Snapshot

The table below is generated from the current run; no stale metrics from earlier data-loading or encoding pipelines are hard-coded.

In [21]:
results = pd.DataFrame(
    {
        "Native CatBoost PR-AUC": [
            catboost_native_pr_auc,
            native_test_pr_auc,
        ],
        "OrderedTargetEncoder PR-AUC": [
            ordered_te_pr_auc,
            ote_test_pr_auc,
        ],
    },
    index=["validation", "test"],
)
results["absolute_delta"] = (
    results["OrderedTargetEncoder PR-AUC"]
    - results["Native CatBoost PR-AUC"]
)
results["relative_delta_pct"] = (
    results["OrderedTargetEncoder PR-AUC"]
    / results["Native CatBoost PR-AUC"]
    - 1
) * 100

results

,Native CatBoost PR-AUC,OrderedTargetEncoder PR-AUC,absolute_delta,relative_delta_pct
validation,0.227781,0.264605,0.036824,16.166257
test,0.307183,0.319621,0.012438,4.049040


## 10. Export Processed Splits

The final engineered chronological splits are intentionally written to:
- `data/processed/train.parquet`
- `data/processed/val.parquet`
- `data/processed/test.parquet`

These files contain the engineered pre-OTE feature matrix plus `Is Laundering`.

In [22]:
def save_dataset_split(
    X_split: pd.DataFrame,
    y_split: pd.Series,
    output_path: Path,
) -> Path:
    if len(X_split) != len(y_split):
        raise ValueError("The number of rows X and y do not match")

    dataset = X_split.reset_index(drop=True).copy()
    dataset["Is Laundering"] = y_split.reset_index(drop=True).astype("int8")
    dataset.to_parquet(output_path, index=False)
    return output_path


SPLIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
saved_paths = {
    "train": save_dataset_split(
        X_train, y_train, SPLIT_OUTPUT_DIR / "train.parquet"
    ),
    "val": save_dataset_split(
        X_val, y_val, SPLIT_OUTPUT_DIR / "val.parquet"
    ),
    "test": save_dataset_split(
        X_test, y_test, SPLIT_OUTPUT_DIR / "test.parquet"
    ),
}
display(pd.Series(saved_paths, name="saved to"))

train    /Users/artem/Documents/Projects/AML/data/proce...
val      /Users/artem/Documents/Projects/AML/data/proce...
test     /Users/artem/Documents/Projects/AML/data/proce...
Name: saved to, dtype: object